## LLM

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
# from config import MODEL_NAME
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)


Loading weights: 100%|██████████| 338/338 [00:00<00:00, 2062.39it/s]
Some parameters are on the meta device because they were offloaded to the cpu and disk.


In [20]:
messages = [
    {"role": "system", "content": "You are a helpful assistant that can answer questions about the question provided."},
    {"role": "user", "content": "What is the national animal of india?"}
]

text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

In [21]:
print(text)

<|im_start|>system
You are a helpful assistant that can answer questions about the question provided.<|im_end|>
<|im_start|>user
What is the national animal of india?<|im_end|>
<|im_start|>assistant



In [22]:
inputs = tokenizer(text, return_tensors="pt").to(model.device)

In [23]:
inputs

{'input_ids': tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,    429,
            646,   4226,   4755,    911,    279,   3405,   3897,     13, 151645,
            198, 151644,    872,    198,   3838,    374,    279,   5313,   9864,
            315,  27711,     30, 151645,    198, 151644,  77091,    198]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [24]:
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=False,
    temperature=0.3
)

In [25]:
outputs

tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,    429,
            646,   4226,   4755,    911,    279,   3405,   3897,     13, 151645,
            198, 151644,    872,    198,   3838,    374,    279,   5313,   9864,
            315,  27711,     30, 151645,    198, 151644,  77091,    198,    785,
           5313,   9864,    315,   6747,    374,    279,   7748,  78089,     13,
         151645]])

In [26]:
inputs["input_ids"].shape[1]

35

In [27]:
generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

In [28]:
generated_tokens

tensor([   785,   5313,   9864,    315,   6747,    374,    279,   7748,  78089,
            13, 151645])

In [29]:
tokenizer.decode(generated_tokens, skip_special_tokens=True)

'The national animal of India is the Indian Elephant.'

## VectorStore

In [31]:
import faiss
import numpy as np
import pickle
# from config import FAISS_INDEX_PATH
FAISS_INDEX_PATH = "faiss_index"

In [ ]:
index = faiss.IndexFlatL2()